# Custom Content Filters

This notebook demonstrates how to build **reusable content filters** that can be composed into guardrail pipelines.

Key concepts:
- **Severity levels** control what happens when a filter matches: BLOCK, WARN, or REDACT
- **Filters are composable** — run multiple filters in sequence and stop at the first violation
- The **base class pattern** makes it easy to add new filter types

These filter classes are the guardrail-specific building blocks used throughout this tutorial; each notebook defines them inline so it runs standalone.

## Filter Pipeline Architecture

<div style="text-align:center">
    <img src="images/filter_pipeline.png" width="85%" />
</div>

## Setup

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools --upgrade -q

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Optional
import re

## Data Models: Severity and FilterResult

Every filter evaluation returns a `FilterResult` that tells the guardrail what action to take:
- `Severity.BLOCK` — reject the entire request/response
- `Severity.WARN` — log a warning but allow through
- `Severity.REDACT` — remove/replace the matched content

In [ ]:
class Severity(Enum):
    """Action to take when a filter matches."""
    BLOCK = "block"
    WARN = "warn"
    REDACT = "redact"


@dataclass
class FilterResult:
    """Result of a content filter evaluation."""
    passed: bool
    filter_name: str
    severity: Severity
    message: Optional[str] = None
    redacted_text: Optional[str] = None

## Base Class: ContentFilter

All content filters inherit from this base class. Subclasses must override the `evaluate()` method to implement custom filtering logic.

In [ ]:
class ContentFilter:
    """Base class for content filters.

    Subclasses must override the `evaluate()` method to implement
    custom filtering logic.
    """

    def __init__(self, name: str, severity: Severity = Severity.BLOCK):
        self.name = name
        self.severity = severity

    def evaluate(self, text: str) -> FilterResult:
        """Evaluate text against this filter. Override in subclasses."""
        raise NotImplementedError("Subclasses must implement evaluate()")

## RegexContentFilter

Uses regex pattern matching to detect structured sensitive information like emails, phone numbers, and SSNs. When severity is `REDACT`, matched patterns are replaced with `[REDACTED]`.

In [ ]:
class RegexContentFilter(ContentFilter):
    """Content filter using regex pattern matching."""

    def __init__(self, name: str, patterns: list[str], severity: Severity = Severity.BLOCK):
        super().__init__(name, severity)
        self.patterns = [re.compile(p) for p in patterns]

    def evaluate(self, text: str) -> FilterResult:
        for pattern in self.patterns:
            if pattern.search(text):
                if self.severity == Severity.REDACT:
                    redacted = text
                    for p in self.patterns:
                        redacted = p.sub("[REDACTED]", redacted)
                    return FilterResult(
                        passed=False,
                        filter_name=self.name,
                        severity=self.severity,
                        message=f"Pattern matched: {pattern.pattern}",
                        redacted_text=redacted,
                    )
                return FilterResult(
                    passed=False,
                    filter_name=self.name,
                    severity=self.severity,
                    message=f"Pattern matched: {pattern.pattern}",
                )
        return FilterResult(passed=True, filter_name=self.name, severity=self.severity)

## KeywordContentFilter

Performs case-insensitive matching against a list of prohibited keywords or phrases. Useful for blocking off-topic requests or detecting harmful content categories.

In [ ]:
class KeywordContentFilter(ContentFilter):
    """Content filter using keyword-based topic detection."""

    def __init__(self, name: str, keywords: list[str], severity: Severity = Severity.BLOCK):
        super().__init__(name, severity)
        self.keywords = [kw.lower() for kw in keywords]

    def evaluate(self, text: str) -> FilterResult:
        text_lower = text.lower()
        for keyword in self.keywords:
            if keyword in text_lower:
                return FilterResult(
                    passed=False,
                    filter_name=self.name,
                    severity=self.severity,
                    message=f"Prohibited keyword detected: '{keyword}'",
                )
        return FilterResult(passed=True, filter_name=self.name, severity=self.severity)

## FormatComplianceFilter

Validates output format compliance — ensures responses don't include code execution instructions or other format violations. This is an example of a domain-specific filter.

In [ ]:
class FormatComplianceFilter(ContentFilter):
    """Content filter that validates output format compliance."""

    EXECUTION_PATTERNS = [
        re.compile(r"\b(run|execute|eval)\s*\(", re.IGNORECASE),
        re.compile(r"```\s*(bash|shell|sh)\b", re.IGNORECASE),
        re.compile(r"\$\s*\w+"),  # Shell variable references
        re.compile(r"sudo\s+\w+", re.IGNORECASE),
    ]

    def __init__(self, name: str = "format_compliance", severity: Severity = Severity.BLOCK):
        super().__init__(name, severity)

    def evaluate(self, text: str) -> FilterResult:
        for pattern in self.EXECUTION_PATTERNS:
            if pattern.search(text):
                return FilterResult(
                    passed=False,
                    filter_name=self.name,
                    severity=self.severity,
                    message=f"Output contains code execution instruction: {pattern.pattern}",
                )
        return FilterResult(passed=True, filter_name=self.name, severity=self.severity)

## Pipeline Helper: `run_filters()`

Evaluates text against a list of filters in order. The first filter that fails has its result returned immediately. If all filters pass, returns `None`.

In [ ]:
def run_filters(text: str, filters: list[ContentFilter]) -> Optional[FilterResult]:
    """Evaluate text against a list of filters, returning the first violation.

    Args:
        text: The text to evaluate.
        filters: Ordered list of content filters to apply.

    Returns:
        The FilterResult of the first failing filter, or None if all pass.
    """
    for content_filter in filters:
        result = content_filter.evaluate(text)
        if not result.passed:
            return result
    return None

## Demo: RegexContentFilter (PII Detection with REDACT)

In [ ]:
pii_filter = RegexContentFilter(
    name="pii_detector",
    patterns=[
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",  # Email
        r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",  # Phone number
        r"\b\d{3}-\d{2}-\d{4}\b",  # SSN
    ],
    severity=Severity.REDACT,
)

test_text = "Contact me at john@example.com or call 555-123-4567"
result = pii_filter.evaluate(test_text)
print(f"Input:    {test_text}")
print(f"Passed:   {result.passed}")
print(f"Severity: {result.severity.value}")
print(f"Message:  {result.message}")
print(f"Redacted: {result.redacted_text}")

## Demo: KeywordContentFilter (Topic Blocking)

In [ ]:
topic_filter = KeywordContentFilter(
    name="topic_blocker",
    keywords=["hack", "exploit", "bypass security"],
    severity=Severity.BLOCK,
)

# Blocked text
blocked_text = "How do I hack into a system?"
result = topic_filter.evaluate(blocked_text)
print(f"Input:   {blocked_text}")
print(f"Passed:  {result.passed}")
print(f"Message: {result.message}")

# Safe text
safe_text = "How do I set up a firewall?"
result = topic_filter.evaluate(safe_text)
print(f"\nInput:   {safe_text}")
print(f"Passed:  {result.passed}")

## Demo: FormatComplianceFilter (Output Validation)

In [ ]:
format_filter = FormatComplianceFilter()

# Unsafe output with code execution instruction
unsafe_output = "To fix this, run: sudo rm -rf /tmp/cache"
result = format_filter.evaluate(unsafe_output)
print(f"Input:   {unsafe_output}")
print(f"Passed:  {result.passed}")
print(f"Message: {result.message}")

# Safe output
safe_output = "The recommended approach is to clear the cache manually."
result = format_filter.evaluate(safe_output)
print(f"\nInput:   {safe_output}")
print(f"Passed:  {result.passed}")

## Demo: Pipeline Evaluation with `run_filters()`

In [ ]:
filters = [topic_filter, pii_filter, format_filter]

# Text that violates the keyword filter (first in the list)
violation_text = "How to exploit a vulnerability? Email me at attacker@evil.com"
pipeline_result = run_filters(violation_text, filters)
print(f"Input:    {violation_text}")
print(f"First violation from: {pipeline_result.filter_name}")
print(f"Message:  {pipeline_result.message}")

# Clean text that passes all filters
clean_text = "What are best practices for application security?"
pipeline_result = run_filters(clean_text, filters)
print(f"\nInput:    {clean_text}")
print(f"Result:   {'All filters passed' if pipeline_result is None else 'Violation found'}")

## Summary

In this notebook you learned:
1. The `Severity` enum and `FilterResult` dataclass that drive guardrail behavior
2. The `ContentFilter` base class pattern for building custom filters
3. `RegexContentFilter` — pattern-based detection with optional redaction
4. `KeywordContentFilter` — case-insensitive keyword/phrase blocking
5. `FormatComplianceFilter` — domain-specific output validation
6. `run_filters()` — composing filters into a pipeline

**Next Steps:** See `04_guardrail_plugin.ipynb` to learn how to package guardrails as a reusable Plugin.